In [1]:
import numpy as np 
import time

import sys
sys.path.append('../')
from reachy import Reachy, parts
from behavior.manipulate_flyer import Manipulate_flyer

In [2]:
reachy = Reachy(
    head=parts.Head(io='/dev/ttyUSB*'),
    right_arm=parts.RightArm(io='/dev/ttyUSB*', hand='force_gripper'),
    left_arm=parts.LeftArm(io='/dev/ttyUSB*', hand='flyer_hand')
)

In [3]:
manip = Manipulate_flyer(reachy)

In [4]:
def compliant(reachy):
    for m in reachy.right_arm.motors:
        m.compliant = True
    for m in reachy.head.motors:
        m.compliant = True
    for m in reachy.left_arm.motors:
        m.compliant = True
    reachy.head.compliant = True

In [5]:
def stiff(reachy):
    for m in reachy.right_arm.motors:
        m.compliant = False
    for m in reachy.head.motors:
        m.compliant = False
    for m in reachy.left_arm.motors:
        m.compliant = False
    reachy.head.compliant = False

In [6]:
for m in reachy.right_arm.motors:
    m.compliant = True 

In [7]:
reachy.right_arm.hand.gripper.goal_position = 90

## Left arm

In [24]:
current_position = [m.present_position for m in reachy.left_arm.motors]
current_position

[-8.11, 41.165, -75.121, -88.571, 26.54, 37.231]

In [29]:
#goal_position = [-14, 50, -76, -97, 32, 37]
goal_position = [-8.11, 41.165, -75.121, -88.571, 26.54, 37.231]

In [30]:
for m in reachy.left_arm.motors:
    m.compliant = False

In [31]:
time.sleep(0.2)

reachy.goto({
        m.name: j
        for j, m in zip(goal_position, reachy.left_arm.motors)
    }, duration=3, wait=True, interpolation_mode='minjerk')

In [33]:
for m in reachy.left_arm.motors:
    m.compliant = True

In [32]:
current_position = [m.present_position for m in reachy.right_arm.motors]
print(np.round(current_position,2))

[ -10.75  -39.06   70.99 -112.66 -112.76   17.98  -29.76  -44.43]


In [34]:
current_position

[-10.747, -39.055, 70.989, -112.659, -112.757, 17.978, -29.765, -44.428]

In [5]:
#right_arm_step1 = [-14, -45, 67, -117, 99, 28, -33, -32]
right_arm_step1 = [1, -66, 63, -124, -99, 0, 11, -17]
right_arm_pos = [-18, -33, 66, -117, 98, 21, -34, -34]

#[  -8.2   -38.79   68.88 -112.75  -98.97   13.23  -14.52  -15.4 ]

base_pos_right = [1, -30, 64, -70, 99, -9, 23, 17]

In [6]:
for m in reachy.right_arm.motors:
    m.compliant = False

In [7]:
time.sleep(0.2)

reachy.goto({
        m.name: j
        for j, m in zip(right_arm_step1, reachy.right_arm.motors)
    }, duration=3, wait=True, interpolation_mode='minjerk')

In [19]:
for m in reachy.right_arm.motors:
    m.compliant = True

## Test right arm

In [8]:
for m in reachy.right_arm.motors:
    m.compliant = False

In [24]:
right_arm_step1 = [1, -66, 63, -124, -99, 0, 11, -17]

right_arm_step2 = [ -12.5,   -37.91,   62.55, -114.5,    -98.42,   30.2 ,  -36.51 , -30.06]

base_pos_right = [1, -30, 64, -70, -99, -9, 23, 17]

A = reachy.right_arm.forward_kinematics(joints_position=right_arm_step2)
A[2][3] -= 0.05

JA = reachy.right_arm.inverse_kinematics(A,q0=right_arm_step2)
print(np.round(JA,2))

B = A.copy()
B[1][3] -= 0.05

JB = reachy.right_arm.inverse_kinematics(B,q0=right_arm_step2)
print(np.round(JB,2))

C = B.copy()
C[1][3] -= 0.1

JC = reachy.right_arm.inverse_kinematics(C,q0=right_arm_step2)
JC[7] = 90
print(np.round(JC,2))

[ -11.09  -35.64   65.21 -104.47  -90.62   28.29  -26.56  -30.06]
[ -10.25  -45.04   62.74 -106.8   -92.91   30.04  -20.98  -30.06]
[  -6.25  -61.34   58.52 -107.1   -85.93   30.38   -1.18   90.  ]


In [18]:
reachy.goto({
        m.name: j
        for j, m in zip(base_pos_right, reachy.right_arm.motors)
    }, duration=2, wait=True, interpolation_mode='minjerk')

time.sleep(0.2)

# reachy.goto({
#         m.name: j
#         for j, m in zip(right_arm_step1, reachy.right_arm.motors)
#     }, duration=2, wait=True, interpolation_mode='minjerk')

# reachy.goto({
#         m.name: j
#         for j, m in zip(right_arm_step2, reachy.right_arm.motors)
#     }, duration=1, wait=True, interpolation_mode='minjerk')

# reachy.goto({
#         m.name: j
#         for j, m in zip(JA, reachy.right_arm.motors)
#     }, duration=1, wait=True, interpolation_mode='minjerk')

# reachy.goto({
#         m.name: j
#         for j, m in zip(JB, reachy.right_arm.motors)
#     }, duration=1, wait=True, interpolation_mode='minjerk')

# reachy.goto({
#     'right_arm.hand.gripper': 90,
# }, duration = 1, wait=True)

# reachy.goto({
#         m.name: j
#         for j, m in zip(JC, reachy.right_arm.motors)
#     }, duration=2, wait=True, interpolation_mode='minjerk')

In [11]:
reachy.goto({        
        'right_arm.shoulder_pitch': -25,
        'right_arm.shoulder_roll': -10,
        'right_arm.arm_yaw': 24,    
        'right_arm.elbow_pitch': -100,
        'right_arm.hand.forearm_yaw': -140,
        'right_arm.hand.wrist_pitch': 0,
        'right_arm.hand.wrist_roll': -40,
        }, duration=2,wait=True,starting_point='goal_position', interpolation_mode='minjerk'),

([<reachy.trajectory.interpolation.MinimumJerk at 0xa357e030>,
  <reachy.trajectory.interpolation.MinimumJerk at 0xa357e4f0>],)

In [17]:
for m in reachy.right_arm.motors:
    m.compliant = True

### Test whole movement

In [3]:
for m in reachy.right_arm.motors:
    m.compliant = False
for m in reachy.left_arm.motors:
    m.compliant = False

In [6]:
goal_position = [-14, 50, -76, -97, 32, 37]

right_arm_step1 = [1, -66, 63, -124, -99, 0, 11, -17]

right_arm_step2 = [ -12.5,   -37.91,   62.55, -114.5,    -98.42,   30.2 ,  -36.51 , -30.06]

base_pos_right = [1, -30, 64, -70, -99, -9, 23, 17]

A = reachy.right_arm.forward_kinematics(joints_position=right_arm_step2)
A[2][3] -= 0.05

JA = reachy.right_arm.inverse_kinematics(A,q0=right_arm_step2)
print(np.round(JA,2))

B = A.copy()
B[1][3] -= 0.05

JB = reachy.right_arm.inverse_kinematics(B,q0=right_arm_step2)
print(np.round(JB,2))

C = B.copy()
C[1][3] -= 0.1

JC = reachy.right_arm.inverse_kinematics(C,q0=right_arm_step2)
JC[7] = 90
print(np.round(JC,2))

[  -9.51  -34.79   64.19 -105.27   98.6    33.54  -44.57  -30.06]
[  -7.66  -45.83   61.09 -108.19   95.74   34.18  -45.    -30.06]
[  -1.01  -66.9    55.13 -110.27   90.4    37.99  -45.     90.  ]


In [7]:
reachy.goto({
        m.name: j
        for j, m in zip(goal_position, reachy.left_arm.motors)
    }, duration=3, wait=True, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(base_pos_right, reachy.right_arm.motors)
    }, duration=2, wait=True, interpolation_mode='minjerk')

time.sleep(0.2)

reachy.goto({
        m.name: j
        for j, m in zip(right_arm_step1, reachy.right_arm.motors)
    }, duration=2, wait=True, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(right_arm_step2, reachy.right_arm.motors)
    }, duration=1, wait=True, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(JA, reachy.right_arm.motors)
    }, duration=1, wait=True, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(JB, reachy.right_arm.motors)
    }, duration=1, wait=True, interpolation_mode='minjerk')

reachy.goto({
    'right_arm.hand.gripper': 90,
}, duration = 1, wait=True)

reachy.goto({
        m.name: j
        for j, m in zip(JC, reachy.right_arm.motors)
    }, duration=2, wait=True, interpolation_mode='minjerk')

KeyboardInterrupt: 

In [166]:
reachy.goto({        
        'right_arm.shoulder_pitch': -25,
        'right_arm.shoulder_roll': -10,
        'right_arm.arm_yaw': 24,    
        'right_arm.elbow_pitch': -100,
        'right_arm.hand.forearm_yaw': -140,
        'right_arm.hand.wrist_pitch': 0,
        'right_arm.hand.wrist_roll': -40,
        }, duration=2,wait=True,starting_point='goal_position', interpolation_mode='minjerk'),

([<reachy.trajectory.interpolation.MinimumJerk at 0xa0c665f0>,
  <reachy.trajectory.interpolation.MinimumJerk at 0xa0c66f50>],)

In [8]:
for m in reachy.right_arm.motors:
    m.compliant = True

In [9]:
for m in reachy.left_arm.motors:
    m.compliant = True